# 🧑‍💼 HR CV Analyzer — Full Test on Colab

**Steps:**
1. Install dependencies
2. Upload your files (`main.py`, `index.html`)
3. Start FastAPI with ngrok tunnel
4. Open the UI link and test!

> ✅ **MOCK MODE** is on by default — no GPU/model needed to test the UI and API flow.
> Set `USE_MOCK=false` in Cell 4 only after your fine-tuned model is ready.

In [1]:
# ── CELL 1 : Install all dependencies ────────────────────────────────────────

# ── Fix torchao FIRST (before any other imports) ──
!pip install -q "torchao>=0.16.0"
!pip install -q fastapi uvicorn pdfplumber python-multipart pyngrok nest_asyncio peft transformers
# Optional OCR support (uncomment if needed)
# !apt-get install -qq tesseract-ocr tesseract-ocr-fra
# !pip install -q pytesseract pdf2image

print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.7 MB/s eta 0:00:00
✅ Dependencies installed


In [5]:
# ── CELL 2 : Upload main.py and index.html ───────────────────────────────────
from google.colab import files

print('📂 Upload your main.py and index.html files:')
uploaded = files.upload()

import shutil, os
# Make sure both files are at /content/
for fname in uploaded:
    dest = f'/content/{fname}'
    if not os.path.exists(dest):
        shutil.copy(fname, dest)
    print(f'  ✅ {fname} → {dest}')

# Verify
print('\n📋 Files at /content:')
for f in ['main.py', 'index.html']:
    status = '✅' if os.path.exists(f'/content/{f}') else '❌ MISSING'
    print(f'  {status}  {f}')

📂 Upload your main.py and index.html files:


Saving index.html to index (2).html
Saving main.py to main (2).py
  ✅ index (2).html → /content/index (2).html
  ✅ main (2).py → /content/main (2).py

📋 Files at /content:
  ✅  main.py
  ✅  index.html


In [6]:
# ── CELL 3 : Configure ngrok ─────────────────────────────────────────────────
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3CoMdqc5CRFdNjOnvEi4RcmBTfB_2yYHiCw7VYmSBtiqTQDbW'  # 👈 Paste your token here

from pyngrok import ngrok, conf

if NGROK_TOKEN:
    # Changed set_authtoken to set_auth_token (added the underscore)
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok token set')
else:
    print('⚠️  No token set — ngrok will still work but may be rate-limited')
    print('   Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken')

✅ ngrok token set


In [7]:
# ── CELL 4 : Start server + wait for it to be ready, THEN open tunnel ────────
import os, nest_asyncio, uvicorn, threading, time, requests
from pyngrok import ngrok

os.environ['USE_MOCK'] = 'false'   # change to 'false' for real model
os.environ['BASE_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['FINETUNED_MODEL_PATH'] = '/content/drive/MyDrive/finetuned_qwen'

ngrok.kill()
nest_asyncio.apply()

import sys
sys.path.insert(0, '/content')
if 'main' in sys.modules:
    del sys.modules['main']

from main import app

PORT = 8000

def run():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

t = threading.Thread(target=run, daemon=True)
t.start()

# ── Wait until the server actually responds ────────────────────────────────
print('⏳ Waiting for server to be ready...')
for i in range(60):           # wait up to 60 seconds
    try:
        r = requests.get(f'http://localhost:{PORT}/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server is up! ({i+1}s)')
            break
    except Exception:
        pass
    time.sleep(1)
    print(f'  still loading... ({i+1}s)', end='\r')
else:
    print('❌ Server did not start in time. Check the logs above.')
    raise RuntimeError('Server startup timeout')

# ── NOW open the tunnel ────────────────────────────────────────────────────
public_url = ngrok.connect(PORT).public_url

print('\n' + '='*60)
print(f'🚀  API is LIVE at:  {public_url}')
print(f'🖥️   UI is at:        {public_url}/ui')
print(f'📖  Docs:            {public_url}/docs')
print('='*60)

⏳ Waiting for server to be ready...
INFO:     127.0.0.1:58902 - "GET /health HTTP/1.1" 200 OK


INFO:     Started server process [3618]
INFO:     Waiting for application startup.


✅ Server is up! (1s)

🚀  API is LIVE at:  https://quintet-tipoff-sudden.ngrok-free.dev
🖥️   UI is at:        https://quintet-tipoff-sudden.ngrok-free.dev/ui
📖  Docs:            https://quintet-tipoff-sudden.ngrok-free.dev/docs


In [ ]:
# ── CELL 5 : Quick API test (run after Cell 4) ───────────────────────────────
import requests

# Replace with your actual ngrok URL from Cell 4 output
BASE = public_url

# --- Test 1: Health check ---
r = requests.get(f'{BASE}/health')
print('Health:', r.json())

# --- Test 2: Text analysis ---
sample_cv = """
John Doe — Software Engineer
Experience: 3 years Python, FastAPI, Machine Learning, scikit-learn, SQL
Education: BSc Computer Science, University of Algiers
Languages: French, English, Arabic
"""

sample_job = """
We are looking for a Python Backend Developer with experience in:
- FastAPI or Django REST
- PostgreSQL / SQL databases
- Docker and CI/CD
- Machine learning is a plus
- 2+ years of professional experience
"""

r = requests.post(f'{BASE}/analyze', json={
    'cv_text': sample_cv,
    'job_description': sample_job
})

result = r.json()
print('\n📊 Analysis Result:')
print(f'  Score    : {result["score"]}')
print(f'  Verdict  : {result["verdict"]}')
print(f'  Strengths: {result["strengths"]}')
print(f'  Weaknesses: {result["weaknesses"]}')

INFO:     34.16.187.178:0 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'mode': 'mock'}
INFO:     34.16.187.178:0 - "POST /analyze HTTP/1.1" 200 OK

📊 Analysis Result:
  Score    : 10%
  Verdict  : Candidate does not meet the key requirements for this role
  Strengths: ['Candidate has a general professional background relevant to this field']
  Weaknesses: ['Missing key requirement: fastapi or django rest, postgresql / sql databases, docker and ci/cd', 'Also missing: machine learning is a plus, 2+ years of professional experience']


In [ ]:
# ── CELL 6 : Test PDF upload ─────────────────────────────────────────────────
from google.colab import files as colab_files
import requests

print('📎 Upload a CV PDF to test the /analyze/pdf endpoint:')
pdf_uploaded = colab_files.upload()

if pdf_uploaded:
    pdf_name = list(pdf_uploaded.keys())[0]
    pdf_bytes = pdf_uploaded[pdf_name]

    job_desc = 'Python developer with FastAPI, SQL, Docker, and machine learning experience'

    r = requests.post(
        f'{public_url}/analyze/pdf',
        files={'cv_pdf': (pdf_name, pdf_bytes, 'application/pdf')},
        data={'job_description': job_desc}
    )

    if r.status_code == 200:
        result = r.json()
        print(f'\n📊 PDF Analysis Result:')
        print(f'  Extraction : {result["extraction_method"]}')
        print(f'  Score      : {result["score"]}')
        print(f'  Verdict    : {result["verdict"]}')
        for s in result['strengths']:  print(f'  ✓ {s}')
        for w in result['weaknesses']: print(f'  ! {w}')
    else:
        print(f'❌ Error {r.status_code}: {r.text}')
else:
    print('No file uploaded.')

📎 Upload a CV PDF to test the /analyze/pdf endpoint:


Saving Healthcare_Professional_CV.pdf to Healthcare_Professional_CV.pdf
INFO:     34.125.237.146:0 - "POST /analyze/pdf HTTP/1.1" 200 OK

📊 PDF Analysis Result:
  Extraction : text
  Score      : 10%
  Verdict    : Candidate does not meet the key requirements for this role
  ✓ Additional background not required but valuable: nurse with over
  ! Missing key requirement: python developer with fastapi, sql docker and machine, learning experience


In [1]:
# ── CELL 7 : Switch to REAL model mode (after fine-tuning is done) ────────────
# Only run this cell when your fine-tuned model is saved to Google Drive

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Verify model files exist
import os
MODEL_PATH = '/content/drive/MyDrive/finetuned_qwen'  # adjust if different
required = ['adapter_config.json', 'adapter_model.safetensors']
for f in required:
    full = os.path.join(MODEL_PATH, f)
    status = '✅' if os.path.exists(full) else '❌ MISSING'
    print(f'  {status}  {f}')

# 3. Switch to real mode and restart
os.environ['USE_MOCK'] = 'false'
os.environ['FINETUNED_MODEL_PATH'] = MODEL_PATH
print('\n⚠️  Now re-run Cell 4 to restart the server in REAL model mode.')
print('    (The model will load — this takes ~2 min on T4 GPU)')

Mounted at /content/drive
  ✅  adapter_config.json
  ✅  adapter_model.safetensors

⚠️  Now re-run Cell 4 to restart the server in REAL model mode.
    (The model will load — this takes ~2 min on T4 GPU)
